In [2]:
#!pip install --update entsoe-py 
from entsoe import EntsoePandasClient
from entsoe.exceptions import NoMatchingDataError, PaginationError
from requests.exceptions import HTTPError

import pandas as pd
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from pathlib import Path

API_KEY = "1206c24a-2fb6-4ec3-a47f-0e1be10dacd4"
client = EntsoePandasClient(api_key= API_KEY)

In [ ]:
start = pd.Timestamp("2024-01-01", tz="Europe/Brussels")
end   = pd.Timestamp("2025-01-01", tz="Europe/Brussels")

country_code = "NL"

process_types = {
    "A51": "aFRR",
    "A47": "mFRR",
    "A46": "RR"
}
contract_terms = ["A01"]  # daily

available = []
missing = []
frames = []

#loop process type per contract term
for pt, pname in process_types.items():
    for term in contract_terms:
        try:
            df = client.query_contracted_reserve_prices(
                country_code= country_code,
                start=start, end=end,
                process_type=pt,
                type_marketagreement_type=term,
                psr_type=None
            )
            if df.empty:
                missing.append((pname, pt, term, "empty frame"))
                continue

            if isinstance(df, pd.Series):
                df = df.to_frame("price_eur_per_mw_term")
            else:
                numcol = "value" if "value" in df.columns else df.columns[0]
                df = df.rename(columns={numcol: "price_eur_per_mw_term"})[["price_eur_per_mw_term"]]

            df = df[~df.index.duplicated(keep="last")]
            df.index.name = "period_start"
            df["product"] = pname
            df["process_type"] = pt
            df["term_code"] = term
            frames.append(df)

            available.append((pname, pt, term, df.index.min(), df.index.max()))

        except NoMatchingDataError:
            missing.append((pname, pt, term, "NoMatchingData"))
        except Exception as e:
            missing.append((pname, pt, term, type(e).__name__))

print("AVAILABLE:")
for row in available:
    print(row)

print("\nMISSING:")
for row in missing:
    print(row)


Connection Error, retrying in 10 seconds
